# SymTRELLIS Evaluation

This notebook rebuilds the main comparison and ablation directly from an evaluation workspace. It uses only Python, pandas, the workspace metadata, shape filenames, and score JSON files.

Each table is evaluated on its own largest common-success subset: a sample is retained only when every trial shown in that table has `eval_success=True`. Failed or missing evaluations are not assigned penalty values, and every method within one table is averaged over exactly the same samples.

In [1]:
import json
from pathlib import Path

import pandas as pd

## Configuration

`model_folders` defines the trials used by the main comparison and ablation. SymTRELLIS uses the legacy sparse-structure mapper, the finetuned neighbor-graph shape mapper, and guidance strength 1.0 at both stages.

In [2]:
dataworkspace_dir = Path("/mnt/scratch/eval_workspace_after_acceptance")

model_folders = {
    "TripoSG": "reference_scores/triposg_seed_43",
    "Hunyuan3D-2.1": "reference_scores/hunyuan3d",
    "TRELLIS 2": "reference_scores/vanilla_trellis2",
    "Closest-point Average": ("experiments/score_postprocess_vanilla_trellis2_symmetry_gt_closest_point_average"),
    "Sector Replication": ("experiments/score_postprocess_vanilla_trellis2_symmetry_gt_sector_replication"),
    "Voxel Majority": ("experiments/score_postprocess_vanilla_trellis2_symmetry_gt_voxel_majority"),
    "Ours": ("experiments/score_shape_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_sparse_structure_swin3d_legacy_sym_old_pred_vc0_sym_old_pred"),
    "Ours w/ gt-symm": ("experiments/score_shape_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"),
    "Vanilla (TRELLIS.2)": "reference_scores/vanilla_trellis2",
    "Sparse structure only": ("experiments/score_shape_s114514_ns0.0_gs0.0_gd0.0_" "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"),
    "Sparse structure and shape": ("experiments/score_shape_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_gs1.0_gd0.3_" "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"),
}

main_models = [
    "Closest-point Average",
    "Sector Replication",
    "Voxel Majority",
    "TripoSG",
    "Hunyuan3D-2.1",
    "TRELLIS 2",
    "Ours",
    "Ours w/ gt-symm",
]

ablation_models = [
    "Vanilla (TRELLIS.2)",
    "Sparse structure only",
    "Sparse structure and shape",
]

## Data loading

Ground-truth folds are reconstructed from the shape filenames using the same point-group convention as the evaluation workspace. Each score JSON is flattened into one row, and `eval_success` defines whether the evaluation is available for comparison.

In [3]:
def read_shape_folds(dataworkspace_dir):
    polyhedral_folds = {"I": 5, "Ih": 5, "O": 4, "Oh": 4, "T": 3, "Td": 3, "Th": 3}
    rows = []

    for shape_path in sorted((dataworkspace_dir / "shapes").glob("*.glb")):
        shape_id_text, group_text, symmetry_axis = shape_path.stem.split("_")
        symmetry_group = group_text[0].upper() + group_text[1:]

        if symmetry_group in polyhedral_folds:
            symmetry_fold = polyhedral_folds[symmetry_group]
        elif symmetry_group == "S1":
            symmetry_fold = 1
        else:
            order = int("".join(character for character in symmetry_group if character.isdigit()))
            symmetry_fold = order // 2 if symmetry_group.startswith("S") else order

        rows.append(
            {
                "shape_id": int(shape_id_text),
                "symmetry_group": symmetry_group,
                "symmetry_axis": symmetry_axis,
                "gt_fold": symmetry_fold,
            }
        )

    return pd.DataFrame(rows).set_index("shape_id")


def read_trial(trial_name, relative_folder, dataworkspace_dir, metadata, shape_folds):
    rows = []

    for score_path in sorted((dataworkspace_dir / relative_folder).glob("*.json")):
        with score_path.open() as file:
            score = json.load(file)

        row = {
            "trial": trial_name,
            "idx": score_path.stem,
            "score_present": True,
            "eval_success": score["eval_success"],
        }

        if score["eval_success"]:
            symmetry = score["self_symm_result"]
            reconstruction = score["reconstruction_result"]
            row.update(
                {
                    "pred_fold": score["pred_fold"],
                    "max_sd": symmetry["max_sd"],
                    "mean_sd": symmetry["mean_sd"],
                    "max_bad_rate_001": symmetry["max_bad_rate"]["0.01"],
                    "max_bad_rate_003": symmetry["max_bad_rate"]["0.03"],
                    "max_bad_rate_01": symmetry["max_bad_rate"]["0.1"],
                    "mean_bad_rate_001": symmetry["mean_bad_rate"]["0.01"],
                    "mean_bad_rate_003": symmetry["mean_bad_rate"]["0.03"],
                    "mean_bad_rate_01": symmetry["mean_bad_rate"]["0.1"],
                    "cd_recon": reconstruction["cd_recon"],
                }
            )

        rows.append(row)

    trial_scores = pd.DataFrame(rows)
    trial_scores = metadata[["idx", "shape_id", "view_id"]].merge(
        trial_scores, on="idx", how="left"
    )
    trial_scores["trial"] = trial_name

    return trial_scores.merge(shape_folds[["gt_fold"]], on="shape_id", how="left")


def summarize_trials(scores, selected_models):
    selected_scores = scores[scores["trial"].isin(selected_models)]
    success_matrix = (
        selected_scores.assign(success=selected_scores["eval_success"].eq(True))
        .pivot(index="idx", columns="trial", values="success")
        .reindex(columns=selected_models, fill_value=False)
        .fillna(False)
    )
    common_ids = success_matrix.index[success_matrix.all(axis=1)]
    common_scores = selected_scores[selected_scores["idx"].isin(common_ids)].copy()
    common_scores["fold_accuracy"] = common_scores["pred_fold"].eq(
        common_scores["gt_fold"]
    )

    metric_columns = [
        "max_sd",
        "mean_sd",
        "max_bad_rate_001",
        "max_bad_rate_003",
        "max_bad_rate_01",
        "mean_bad_rate_001",
        "mean_bad_rate_003",
        "mean_bad_rate_01",
        "fold_accuracy",
        "cd_recon",
    ]
    summary = (
        common_scores.groupby("trial", sort=False)[metric_columns]
        .mean()
        .reindex(selected_models)
    )
    summary["sample_count"] = common_scores.groupby("trial").size().reindex(selected_models)

    scaled_columns = [column for column in metric_columns if column != "fold_accuracy"]
    summary[scaled_columns] *= 1000
    summary["fold_accuracy"] *= 100

    return common_ids, summary

In [4]:
metadata = pd.read_csv(dataworkspace_dir / "metadata.csv", dtype={"idx": str})
shape_folds = read_shape_folds(dataworkspace_dir)
selected_models = list(dict.fromkeys(main_models + ablation_models))
scores = pd.concat(
    [
        read_trial(
            trial_name,
            model_folders[trial_name],
            dataworkspace_dir,
            metadata,
            shape_folds,
        )
        for trial_name in selected_models
    ],
    ignore_index=True,
)

source_summary = (
    scores.assign(
        json_count=scores["score_present"].eq(True),
        successful=scores["eval_success"].eq(True),
        failed=scores["eval_success"].eq(False),
    )
    .groupby("trial", sort=False)[["json_count", "successful", "failed"]]
    .sum()
    .reindex(selected_models)
)
source_summary.insert(0, "relative_folder", [model_folders[name] for name in selected_models])
source_summary.index.name = "Trial"
source_summary

,relative_folder,json_count,successful,failed
Trial,,,,
Closest-point Average,experiments/score_postprocess_vanilla_trellis2...,2120,2077,43
Sector Replication,experiments/score_postprocess_vanilla_trellis2...,2120,2105,15
Voxel Majority,experiments/score_postprocess_vanilla_trellis2...,2120,2078,42
TripoSG,reference_scores/triposg_seed_43,2120,2102,18
Hunyuan3D-2.1,reference_scores/hunyuan3d,2120,2099,21
TRELLIS 2,reference_scores/vanilla_trellis2,2120,2104,16
Ours,experiments/score_shape_s114514_ns0.5_gs1.0_gd...,2120,2108,12
Ours w/ gt-symm,experiments/score_shape_s114514_ns0.5_gs1.0_gd...,2120,2108,12
Vanilla (TRELLIS.2),reference_scores/vanilla_trellis2,2120,2104,16


## Main comparison

SD, error rates, and CD are reported in $\times 10^3$. Fold accuracy is reported as a percentage. The table uses the exact common-success subset of all displayed methods.

In [5]:
main_ids, main_summary = summarize_trials(scores, main_models)

main_subset = pd.DataFrame(
    {
        "Common-success samples": [len(main_ids)],
        "Total samples": [len(metadata)],
        "Coverage (%)": [100 * len(main_ids) / len(metadata)],
    },
    index=["Main table"],
)
display(main_subset.style.format({"Coverage (%)": "{:.2f}"}))

main_table = main_summary.rename(
    columns={
        "max_sd": "Max SD",
        "mean_sd": "Mean SD",
        "max_bad_rate_001": "Max Err. @ .01",
        "max_bad_rate_003": "Max Err. @ .03",
        "max_bad_rate_01": "Max Err. @ .1",
        "mean_bad_rate_001": "Mean Err. @ .01",
        "mean_bad_rate_003": "Mean Err. @ .03",
        "mean_bad_rate_01": "Mean Err. @ .1",
        "fold_accuracy": "Fold Acc. (%)",
        "cd_recon": "CD",
    }
)[
    [
        "Max SD",
        "Mean SD",
        "Max Err. @ .01",
        "Max Err. @ .03",
        "Max Err. @ .1",
        "Mean Err. @ .01",
        "Mean Err. @ .03",
        "Mean Err. @ .1",
        "Fold Acc. (%)",
        "CD",
    ]
]
main_table.index.name = "Method"
display(main_table.style.format("{:.3f}"))

,Common-success samples,Total samples,Coverage (%)
Main table,2035,2120,95.99


,Max SD,Mean SD,Max Err. @ .01,Max Err. @ .03,Max Err. @ .1,Mean Err. @ .01,Mean Err. @ .03,Mean Err. @ .1,Fold Acc. (%),CD
Method,,,,,,,,,,
Closest-point Average,1.269,0.734,2.400,1.359,0.634,1.494,0.810,0.425,93.022,68.886
Sector Replication,0.121,0.072,0.629,0.159,0.000,0.339,0.079,0.000,99.459,6.510
Voxel Majority,4.372,2.501,74.726,16.626,4.012,39.421,8.216,1.346,99.165,14.829
TripoSG,0.686,0.330,9.276,1.265,0.058,3.650,0.482,0.020,77.248,3.440
Hunyuan3D-2.1,1.519,0.729,28.663,6.517,0.494,11.874,2.481,0.169,73.956,4.435
TRELLIS 2,1.813,0.853,39.017,9.164,0.725,16.142,3.741,0.283,82.850,4.489
Ours,0.971,0.482,15.261,3.877,0.337,6.719,1.621,0.149,81.032,5.854
Ours w/ gt-symm,0.887,0.421,12.867,2.936,0.267,5.163,1.074,0.070,91.646,6.267


## Ablation

The ablation table uses its own common-success subset. All reported metrics are in $\times 10^3$.

In [6]:
ablation_ids, ablation_summary = summarize_trials(scores, ablation_models)

ablation_subset = pd.DataFrame(
    {
        "Common-success samples": [len(ablation_ids)],
        "Total samples": [len(metadata)],
        "Coverage (%)": [100 * len(ablation_ids) / len(metadata)],
    },
    index=["Ablation table"],
)
display(ablation_subset.style.format({"Coverage (%)": "{:.2f}"}))

ablation_table = ablation_summary.rename(
    columns={
        "max_bad_rate_003": "Max Err. @ .03",
        "mean_bad_rate_003": "Mean Err. @ .03",
        "cd_recon": "CD",
    }
)[["Max Err. @ .03", "Mean Err. @ .03", "CD"]]
ablation_table.index.name = "Method"
display(ablation_table.style.format("{:.3f}"))

,Common-success samples,Total samples,Coverage (%)
Ablation table,2100,2120,99.06


,Max Err. @ .03,Mean Err. @ .03,CD
Method,,,
Vanilla (TRELLIS.2),9.774,3.908,4.645
Sparse structure only,6.864,2.682,7.191
Sparse structure and shape,2.901,1.065,7.148


## Fold-wise analysis

Fold accuracy is computed by GT rotational fold on the exact common-success subset of the five displayed methods. Reflection-only $S_1$ samples are excluded. SD, error rates, and CD are reported in $\times 10^3$ on the rotational samples for which all five methods recover the correct fold.

In [7]:
fold_models = [
    "TripoSG",
    "Hunyuan3D-2.1",
    "TRELLIS 2",
    "Ours",
    "Ours w/ gt-symm",
]
fold_groups = ["2", "3", "4", "5", "6", "7", "8", "9", "10+"]

fold_scores = scores[scores["trial"].isin(fold_models)]
success_matrix = (
    fold_scores.assign(success=fold_scores["eval_success"].eq(True))
    .pivot(index="idx", columns="trial", values="success")
    .reindex(columns=fold_models, fill_value=False)
    .fillna(False)
)
common_success_ids = success_matrix.index[success_matrix.all(axis=1)]
rotational_scores = fold_scores[
    fold_scores["idx"].isin(common_success_ids) & fold_scores["gt_fold"].ge(2)
].copy()
rotational_scores["fold_group"] = rotational_scores["gt_fold"].astype(int).astype(str)
rotational_scores.loc[rotational_scores["gt_fold"].ge(10), "fold_group"] = "10+"
rotational_scores["fold_correct"] = rotational_scores["pred_fold"].eq(
    rotational_scores["gt_fold"]
)

fold_counts = (
    rotational_scores.drop_duplicates("idx")["fold_group"]
    .value_counts()
    .reindex(fold_groups, fill_value=0)
)
fold_accuracy = (
    rotational_scores.pivot_table(
        index="trial",
        columns="fold_group",
        values="fold_correct",
        aggfunc="mean",
    )
    .reindex(index=fold_models, columns=fold_groups)
    .mul(100)
)
fold_accuracy.columns = pd.MultiIndex.from_tuples(
    [
        ("Fold Acc. on Common-success Set (%) ↑", f"{fold} (n={fold_counts[fold]})")
        for fold in fold_groups
    ]
)

fold_correct_matrix = (
    rotational_scores.pivot(index="idx", columns="trial", values="fold_correct")
    .reindex(columns=fold_models, fill_value=False)
    .fillna(False)
)
common_fold_correct_ids = fold_correct_matrix.index[fold_correct_matrix.all(axis=1)]
common_fold_correct_scores = rotational_scores[
    rotational_scores["idx"].isin(common_fold_correct_ids)
]
quality_columns = {
    "max_sd": "Max SD",
    "mean_sd": "Mean SD",
    "max_bad_rate_003": "Max Err. @ .03",
    "mean_bad_rate_003": "Mean Err. @ .03",
    "cd_recon": "CD",
}
fold_correct_quality = (
    common_fold_correct_scores.groupby("trial", sort=False)[list(quality_columns)]
    .mean()
    .reindex(fold_models)
    .rename(columns=quality_columns)
    .mul(1000)
)
fold_correct_quality.columns = pd.MultiIndex.from_product(
    [
        [f"Scores on Common Fold-correct Set (n={len(common_fold_correct_ids)}) ↓"],
        list(quality_columns.values()),
    ]
)

fold_analysis_table = pd.concat([fold_accuracy, fold_correct_quality], axis=1)
fold_analysis_table.index.name = "Method"
display(fold_analysis_table.style.format("{:.3f}"))